# Limpieza y validación de aforos vehiculares HMD — segmentación Usaquén

Issue #22 (F2.1) · Fase 2 - Preparación de datos (ETL) · Semana 3 · Responsable: Samuel Campos
Prerrequisito: F1.1 (exploración, `01_exploracion_aforos_hmd.ipynb`) · Co-requisito: F2.2

Este notebook deja lista la base de aforos HMD para la construcción de matrices O/D, con foco en Usaquén:

1. Eliminación de registros duplicados
2. Verificación de consistencia temporal (PERIODO AM/PM)
3. Criterio para estaciones sin cobertura AM+PM completa
4. Segmentación geográfica: geocodificación de las 336 estaciones y cruce contra el polígono oficial de Usaquén (mismo polígono y fuente que usó F1.3 para la red vial: `data/raw/limites/loca.json`, Secretaría Distrital de Planeación)
5. Documentación de cada decisión de limpieza tomada, para trazabilidad del pipeline

**Nota metodológica:** F1.4 (informe de brechas) había dejado esta geocodificación como *pendiente*, señalándola como "la brecha metodológica más urgente" porque el aforo no trae coordenadas ni localidad — solo texto libre en `LOCALIZACIÓN` (ej. `AC 80 X KR 116A`). Este notebook la resuelve geocodificando cada estación contra OpenStreetMap.

In [1]:
import json
import re
import time
from pathlib import Path

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

RAW_PATH = "../data/raw/a. Base de datos aforos hora de maxima demanda.xlsx"
LIMITES_PATH = "../data/raw/limites/loca.json"
GEOCODE_CACHE_PATH = "../data/processed/geocoding_usaquen_cache.json"
OUT_LIMPIO_PATH = "../data/processed/aforos_hmd_limpio.csv"
OUT_USAQUEN_PATH = "../data/processed/aforos_hmd_usaquen.csv"

bd = pd.read_excel(RAW_PATH, sheet_name="BD")
print("Registros originales:", bd.shape)
bd.head(3)

Registros originales: (3956, 7)


,TIPOLOGIA,ID,LOCALIZACIÓN,EODI,PERIODO,SENTIDO,VOLUMEN
0,BICICLETA EN CICLORUTA,BC_3,AC 80 X KR 116A,SI,AM,E-W,1118.0
1,BICICLETA EN CICLORUTA,BC_3,AC 80 X KR 116A,SI,AM,W-E,583.5
2,BICICLETA EN CICLORUTA,BC_3,AC 80 X KR 116A,SI,PM,E-W,472.0


## 1. Eliminación de registros duplicados

La exploración F1.1 ya había identificado 3 pares de filas exactamente duplicadas (todas `BRT`, periodo `AM`, sentido `INGRESO`, estaciones `TM_53`, `TM_6`, `TM_8`). No hay forma de distinguirlas como dos mediciones independientes válidas (mismo ID + mismo periodo + mismo sentido no debería repetirse), así que se eliminan conservando la primera ocurrencia.

**Decisión documentada:** se eliminan duplicados exactos (`drop_duplicates()` sobre todas las columnas). No se eliminan duplicados parciales (mismo ID+PERIODO+SENTIDO con VOLUMEN distinto) porque no se encontró ninguno — se valida explícitamente abajo.

In [2]:
n_dup_exactos = bd.duplicated().sum()
print(f"Duplicados exactos a eliminar: {n_dup_exactos}")

# Verificación: ¿hay duplicados "parciales" (misma clave, distinto VOLUMEN)?
claves = ["TIPOLOGIA", "ID", "LOCALIZACIÓN", "EODI", "PERIODO", "SENTIDO"]
dup_parciales = bd[bd.duplicated(subset=claves, keep=False)]
dup_parciales_reales = dup_parciales[~dup_parciales.duplicated(keep=False) | dup_parciales.duplicated()]
print(f"Filas con misma clave pero distinto VOLUMEN (no son duplicado exacto): "
      f"{dup_parciales.shape[0] - n_dup_exactos * 2}")

bd_dedup = bd.drop_duplicates().reset_index(drop=True)
print("Registros tras eliminar duplicados exactos:", bd_dedup.shape)

Duplicados exactos a eliminar: 3
Filas con misma clave pero distinto VOLUMEN (no son duplicado exacto): 0
Registros tras eliminar duplicados exactos: (3953, 7)


## 2. Consistencia temporal (PERIODO)

El dataset no trae fechas explícitas, solo la franja `PERIODO` (`AM`/`PM`), ya documentado en F1.1. La validación de consistencia temporal se reduce a:

- Confirmar que `PERIODO` solo toma los dos valores válidos del diccionario de datos (`AM`, `PM`).
- Cuantificar estaciones sin cobertura completa AM+PM.

**Decisión documentada (criterio para estaciones incompletas):** no se descartan ni se imputan con promedios sintéticos. Se conserva cada estación con el/los periodo(s) que sí tiene, y se agrega una columna `COBERTURA_PERIODO` (`AM_PM` / `SOLO_AM` / `SOLO_PM`) para que cualquier análisis que compare AM vs PM pueda filtrar explícitamente por `COBERTURA_PERIODO == "AM_PM"`. Se prefiere esto sobre imputar porque introducir un volumen promedio sintético en una estación real sesgaría la matriz O/D con datos que no fueron medidos; y sobre descartar porque perdería ~28% de las estaciones para todo análisis que no dependa de comparar AM vs PM (ej. volumen total por estación).

In [3]:
valores_periodo_validos = {"AM", "PM"}
valores_invalidos = set(bd_dedup["PERIODO"].unique()) - valores_periodo_validos
print("Valores de PERIODO fuera de {AM, PM}:", valores_invalidos)

cobertura = bd_dedup.groupby("ID")["PERIODO"].agg(lambda s: frozenset(s.unique()))
mapa_cobertura = cobertura.apply(
    lambda s: "AM_PM" if s == {"AM", "PM"} else ("SOLO_AM" if s == {"AM"} else "SOLO_PM")
)
bd_dedup["COBERTURA_PERIODO"] = bd_dedup["ID"].map(mapa_cobertura)
print(bd_dedup["COBERTURA_PERIODO"].value_counts())

Valores de PERIODO fuera de {AM, PM}: set()
COBERTURA_PERIODO
AM_PM      3858
SOLO_PM      62
SOLO_AM      33
Name: count, dtype: int64


## 3. Segmentación geográfica: geocodificación y filtro Usaquén

`LOCALIZACIÓN` es texto libre y muy heterogéneo: intersecciones bien formadas (`AC 80 X KR 116A`), variantes sin espacios o con guion (`AK10_X_AC19`, `Suba - Calle 100`), y nombres de estación/portal (`Alquería`, `Toberín`, `Portal del Norte`). Se usan dos métodos, elegidos automáticamente por patrón:

- **Intersección** (contiene `X`/`_X_`/`-` entre dos tokens que lucen como vía): se normalizan las abreviaturas de nomenclatura vial de Bogotá a los nombres que usa OpenStreetMap (`AC`→`Avenida Calle`, `AK`→`Avenida Carrera`, `KR`→`Carrera`, `TV`→`Transversal`, `DG`→`Diagonal`, más un mapa de avenidas con nombre propio como `Avenida Boyacá`, `Avenida NQS`, etc.) y se calcula la **intersección geométrica real** de ambas vías consultando Overpass API (no un simple "buscar el texto", sino la geometría de cada vía y su cruce).
- **Lugar** (nombres de estación/portal sin patrón de intersección): geocodificación de lugar/POI vía Nominatim.

En ambos casos, el resultado se cruza por punto-en-polígono contra el límite oficial de Usaquén (`LocNombre == "USAQUEN"` en `data/raw/limites/loca.json`, Secretaría Distrital de Planeación — el mismo polígono que usó F1.3 para la red OSM).

**Este proceso ya se ejecutó una vez** (llamadas en vivo a Overpass/Nominatim, con límite de 1 solicitud secuencial cada ~2s según sus políticas de uso) y quedó cacheado en `data/processed/geocoding_usaquen_cache.json`. El código de geocodificación completo, para que el proceso sea reproducible, vive en `notebooks/geocoding_usaquen.py`; esta celda solo carga el resultado cacheado.

**Limitación documentada:** esto es geocodificación automática de texto libre con abreviaturas no estandarizadas — no es infalible. Las estaciones que Overpass/Nominatim no pudieron resolver (`in_usaquen is None`) quedan marcadas explícitamente como `SIN_GEOCODIFICAR` en vez de asumirse fuera de Usaquén, y deben revisarse manualmente antes de usarse en la matriz O/D final.

In [4]:
with open(GEOCODE_CACHE_PATH, encoding="utf-8") as f:
    geocode_cache = json.load(f)

print(f"Localizaciones únicas en el aforo: {bd_dedup['LOCALIZACIÓN'].nunique()}")
print(f"Localizaciones geocodificadas (en cache): {len(geocode_cache)}")

def resolver_localidad(loc):
    entry = geocode_cache.get(loc.strip())
    if entry is None:
        return "SIN_GEOCODIFICAR"
    if entry["in_usaquen"] is True:
        return "USAQUEN"
    if entry["in_usaquen"] is False:
        return "OTRA_LOCALIDAD"
    return "SIN_GEOCODIFICAR"

bd_dedup["SEGMENTO_GEOGRAFICO"] = bd_dedup["LOCALIZACIÓN"].apply(resolver_localidad)
resumen_localidad = bd_dedup.drop_duplicates("ID")["SEGMENTO_GEOGRAFICO"].value_counts()
print("\nEstaciones (ID únicos) por segmento geográfico:")
print(resumen_localidad)
print(f"\n% de estaciones geocodificadas con éxito: "
      f"{(1 - resumen_localidad.get('SIN_GEOCODIFICAR', 0) / bd_dedup['ID'].nunique()):.1%}")

Localizaciones únicas en el aforo: 309
Localizaciones geocodificadas (en cache): 308

Estaciones (ID únicos) por segmento geográfico:
SEGMENTO_GEOGRAFICO
OTRA_LOCALIDAD      158
SIN_GEOCODIFICAR    138
USAQUEN              40
Name: count, dtype: int64

% de estaciones geocodificadas con éxito: 58.9%


In [5]:
sin_geocodificar = (
    bd_dedup[bd_dedup["SEGMENTO_GEOGRAFICO"] == "SIN_GEOCODIFICAR"]
    [["ID", "LOCALIZACIÓN"]].drop_duplicates().sort_values("LOCALIZACIÓN")
)
print(f"{len(sin_geocodificar)} estaciones requieren revisión manual (no se pudieron geocodificar automáticamente):")
sin_geocodificar

138 estaciones requieren revisión manual (no se pudieron geocodificar automáticamente):


,ID,LOCALIZACIÓN
282,TPU_50,AC 12 X KR 24
230,TPU_12,AC 13 X AK 68D
1022,TP_54,AC 13 X KR 36
312,TPU_69,AC 147 X AK 9
242,TPU_22,AC 17 X KR 16A
...,...,...
4,BC_5,TV 60 X KR 104
298,TPU_06,TV 60 X KR 91
260,TPU_35,TV 6B E X CL 41 S
3802,TM_11,Terreros - Hospital C.V


In [6]:
usaquen_ids = (
    bd_dedup[bd_dedup["SEGMENTO_GEOGRAFICO"] == "USAQUEN"]
    [["ID", "LOCALIZACIÓN"]].drop_duplicates().sort_values("LOCALIZACIÓN")
)
print(f"{len(usaquen_ids)} estaciones identificadas dentro de Usaquén:")
usaquen_ids

40 estaciones identificadas dentro de Usaquén:


,ID,LOCALIZACIÓN
40,BC_35,AC 116 X AK 15
288,TPU_55,AC 116 X AK 15
56,BC_44,AC 127 X AK 19
290,TPU_56,AC 127 X AK 19
80,BC_47,AC 127 X AK 9
1076,TP_73,AC 134 X AK 9
64,BC_53,AC 134 X AK 9
286,TPU_54,AC 134 X AK 9
1064,TP_67,AC 183 X KR 15
48,BC_42,AC 183 X KR 19


## 4. Consolidado de decisiones de limpieza (trazabilidad)

In [7]:
resumen = {
    "registros_originales": int(bd.shape[0]),
    "duplicados_exactos_eliminados": int(n_dup_exactos),
    "registros_tras_dedup": int(bd_dedup.shape[0]),
    "estaciones_totales": int(bd_dedup["ID"].nunique()),
    "estaciones_solo_am": int((mapa_cobertura == "SOLO_AM").sum()),
    "estaciones_solo_pm": int((mapa_cobertura == "SOLO_PM").sum()),
    "estaciones_am_pm_completo": int((mapa_cobertura == "AM_PM").sum()),
    "estaciones_usaquen": int(resumen_localidad.get("USAQUEN", 0)),
    "estaciones_otra_localidad": int(resumen_localidad.get("OTRA_LOCALIDAD", 0)),
    "estaciones_sin_geocodificar": int(resumen_localidad.get("SIN_GEOCODIFICAR", 0)),
}
resumen

{'registros_originales': 3956,
 'duplicados_exactos_eliminados': 3,
 'registros_tras_dedup': 3953,
 'estaciones_totales': 336,
 'estaciones_solo_am': 33,
 'estaciones_solo_pm': 62,
 'estaciones_am_pm_completo': 241,
 'estaciones_usaquen': 40,
 'estaciones_otra_localidad': 158,
 'estaciones_sin_geocodificar': 138}

**Resumen de decisiones tomadas (para trazabilidad del pipeline):**

1. **Duplicados**: se eliminaron duplicados exactos (fila completa idéntica) vía `drop_duplicates()`. No se encontraron duplicados parciales (misma estación+periodo+sentido con volumen distinto).
2. **Valores faltantes**: no hay `NaN` en el dataset original (confirmado en F1.1). El "vacío" real no es de celdas sino de *cobertura de periodo* por estación (95 de 336 no tienen AM y PM). Criterio adoptado: **no imputar** — se conserva el dato real disponible y se marca con `COBERTURA_PERIODO` para que el consumidor de los datos (construcción de matrices O/D) decida explícitamente si excluye las estaciones incompletas de comparaciones AM vs PM.
3. **Consistencia temporal**: `PERIODO` solo contiene `AM`/`PM`, sin valores inválidos. No hay fechas que validar (el dataset nunca las tuvo, ver F1.1).
4. **Segmentación geográfica (Usaquén)**: se geocodificaron las `LOCALIZACIÓN` únicas contra OpenStreetMap (intersección geométrica real de vías vía Overpass, o lugar/POI vía Nominatim) y se cruzaron contra el polígono oficial de Usaquén. Las estaciones no resueltas automáticamente quedan marcadas `SIN_GEOCODIFICAR` explícitamente en vez de excluirse silenciosamente — requieren revisión manual antes de la matriz O/D final.
5. **Pendiente para el equipo**: revisar manualmente las estaciones `SIN_GEOCODIFICAR` listadas arriba (nombres ambiguos o no indexados en OSM) y confirmar/corregir su segmento geográfico antes de F2.2.

## 5. Exportar datasets limpios

In [8]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)
bd_dedup.to_csv(OUT_LIMPIO_PATH, index=False, encoding="utf-8-sig")

bd_usaquen = bd_dedup[bd_dedup["SEGMENTO_GEOGRAFICO"] == "USAQUEN"].reset_index(drop=True)
bd_usaquen.to_csv(OUT_USAQUEN_PATH, index=False, encoding="utf-8-sig")

print(f"Dataset limpio completo: {bd_dedup.shape} -> {OUT_LIMPIO_PATH}")
print(f"Dataset limpio segmentado a Usaquén: {bd_usaquen.shape} -> {OUT_USAQUEN_PATH}")
print(f"Estaciones únicas en Usaquén: {bd_usaquen['ID'].nunique()}")

Dataset limpio completo: (3953, 9) -> ../data/processed/aforos_hmd_limpio.csv
Dataset limpio segmentado a Usaquén: (278, 9) -> ../data/processed/aforos_hmd_usaquen.csv
Estaciones únicas en Usaquén: 40
